# HW1 Question 2: 2D Dense Matrix Multiplication

Platform: Google Colab, NVIDIA T4. The notebook writes the submitted `matMul.cu` and `benchmark.py`, builds for `sm_75`, validates both data types, generates timing plots, and measures Nsight Compute achieved occupancy.


In [1]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader
!nvcc --version
!ncu --version


Tesla T4, 7.5, 15360 MiB, 580.82.07
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
NVIDIA (R) Nsight Compute Command Line Profiler
Copyright (c) 2018-2025 NVIDIA Corporation
Version 2025.3.1.0 (build 36398880) (public-release)


In [2]:
%%writefile matMul.cu
// Naive row-major C(MxP) = A(MxN) * B(NxP), one CUDA thread per C element.
// Usage: matMul [M N P [double|float]] (defaults: 128 256 32 double)

#include <cuda_runtime.h>

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <memory>
#include <string>
#include <vector>

#define CHECK_CUDA(call)                                                   \
    do {                                                                   \
        cudaError_t err_ = (call);                                         \
        if (err_ != cudaSuccess) {                                         \
            std::fprintf(stderr, "CUDA error '%s' at %s:%d\n",            \
                         cudaGetErrorString(err_), __FILE__, __LINE__);    \
            std::exit(EXIT_FAILURE);                                       \
        }                                                                  \
    } while (0)

constexpr unsigned BLOCK_DIM = 16;
constexpr size_t FULL_VALIDATION_WORK = 1ull << 26;

template <typename T>
__global__ void matMulKernel(const T* __restrict__ A, const T* __restrict__ B,
                             T* __restrict__ C, size_t M, size_t N, size_t P) {
    const size_t col = blockIdx.x * blockDim.x + threadIdx.x;
    const size_t row = blockIdx.y * blockDim.y + threadIdx.y;
    if (row >= M || col >= P) return;

    T acc = T(0);
    const T* aRow = A + row * N;
    for (size_t k = 0; k < N; ++k) acc += aRow[k] * B[k * P + col];
    C[row * P + col] = acc;
}

template <typename T>
static T cpuDot(const T* A, const T* B, size_t row, size_t col, size_t N,
                size_t P) {
    T acc = T(0);
    for (size_t k = 0; k < N; ++k) acc += A[row * N + k] * B[k * P + col];
    return acc;
}

static std::vector<size_t> sampleIndices(size_t n) {
    const size_t candidates[] = {0, 1, n / 4, n / 2 - 1, n / 2,
                                 n / 2 + 1, 3 * n / 4,
                                 n >= 2 ? n - 2 : 0, n - 1};
    std::vector<size_t> indices;
    for (size_t i : candidates) {
        if (i < n) indices.push_back(i);
    }
    std::sort(indices.begin(), indices.end());
    indices.erase(std::unique(indices.begin(), indices.end()), indices.end());
    return indices;
}

static bool checkedMul(size_t a, size_t b, size_t* out) {
    if (a != 0 && b > SIZE_MAX / a) return false;
    *out = a * b;
    return true;
}

static size_t ceilDiv(size_t a, size_t b) { return a / b + (a % b != 0); }

static bool parseDim(const char* s, size_t* out) {
    if (s == nullptr || *s == '\0') return false;
    char* end = nullptr;
    const unsigned long long value = std::strtoull(s, &end, 10);
    if (end == s || *end != '\0' || value == 0 || value > SIZE_MAX) return false;
    *out = value;
    return true;
}

static double nowMs() {
    return std::chrono::duration<double, std::milli>(
               std::chrono::steady_clock::now().time_since_epoch())
        .count();
}

template <typename T>
static int runCase(size_t M, size_t N, size_t P, const char* typeName) {
    size_t aElems, bElems, cElems, work, aBytes, bBytes, cBytes;
    if (!checkedMul(M, N, &aElems) || !checkedMul(N, P, &bElems) ||
        !checkedMul(M, P, &cElems) || !checkedMul(aElems, P, &work) ||
        !checkedMul(aElems, sizeof(T), &aBytes) ||
        !checkedMul(bElems, sizeof(T), &bBytes) ||
        !checkedMul(cElems, sizeof(T), &cBytes)) {
        std::fprintf(stderr, "error: matrix size overflow\n");
        return EXIT_FAILURE;
    }
    if (ceilDiv(M, BLOCK_DIM) > 65535) {
        std::fprintf(stderr, "error: M exceeds CUDA grid.y limit\n");
        return EXIT_FAILURE;
    }

    // new T[n] does not zero-initialize float/double elements.
    std::unique_ptr<T[]> hA(new T[aElems]);
    std::unique_ptr<T[]> hB(new T[bElems]);
    std::unique_ptr<T[]> hC(new T[cElems]);
    T* A = hA.get();
    T* B = hB.get();
    T* C = hC.get();
    // Fixed varying patterns; denominator 16 is exact in float and double.
    for (size_t i = 0; i < aElems; ++i) A[i] = static_cast<T>(i % 17) / T(16);
    for (size_t i = 0; i < bElems; ++i) B[i] = static_cast<T>(i % 13) / T(16);

    T *dA, *dB, *dC;
    CHECK_CUDA(cudaMalloc((void**)&dA, aBytes));
    CHECK_CUDA(cudaMalloc((void**)&dB, bBytes));
    CHECK_CUDA(cudaMalloc((void**)&dC, cBytes));

    const dim3 block(BLOCK_DIM, BLOCK_DIM);
    const dim3 grid(ceilDiv(P, BLOCK_DIM), ceilDiv(M, BLOCK_DIM));

    CHECK_CUDA(cudaDeviceSynchronize());
    const double t0 = nowMs();
    CHECK_CUDA(cudaMemcpy(dA, A, aBytes, cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaMemcpy(dB, B, bBytes, cudaMemcpyHostToDevice));
    CHECK_CUDA(cudaDeviceSynchronize());
    const double t1 = nowMs();

    matMulKernel<T><<<grid, block>>>(dA, dB, dC, M, N, P);
    CHECK_CUDA(cudaGetLastError());
    CHECK_CUDA(cudaDeviceSynchronize());
    const double t2 = nowMs();

    CHECK_CUDA(cudaMemcpy(C, dC, cBytes, cudaMemcpyDeviceToHost));
    CHECK_CUDA(cudaDeviceSynchronize());
    const double t3 = nowMs();

    const bool fullValidation = work <= FULL_VALIDATION_WORK;
    const double tolerance = sizeof(T) == sizeof(double) ? 1e-9 : 1e-3;
    double maxAbs = 0.0, maxRel = 0.0;
    size_t checkedCount = 0;
    bool validationPassed = true;
    auto validate = [&](size_t row, size_t col) {
        const double ref = cpuDot(A, B, row, col, N, P);
        const double absErr = std::fabs(C[row * P + col] - ref);
        const double relErr = absErr / std::max(std::fabs(ref), 1e-30);
        maxAbs = std::max(maxAbs, absErr);
        maxRel = std::max(maxRel, relErr);
        if (absErr > tolerance + tolerance * std::fabs(ref)) validationPassed = false;
        ++checkedCount;
    };
    if (fullValidation) {
        for (size_t row = 0; row < M; ++row)
            for (size_t col = 0; col < P; ++col) validate(row, col);
    } else {
        const auto rows = sampleIndices(M);
        const auto cols = sampleIndices(P);
        for (size_t row : rows)
            for (size_t col : cols) validate(row, col);
    }

    const size_t blocksX = grid.x;
    const size_t blocksY = grid.y;
    const size_t totalThreads = blocksX * blocksY * BLOCK_DIM * BLOCK_DIM;
    const std::string validation =
        fullValidation ? "full" : "sampled:" + std::to_string(checkedCount);
    std::printf(
        "type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,"
        "active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,"
        "validation\n");
    std::printf("%s,%zu,%zu,%zu,%u,%zu,%zu,%zu,%zu,%.4f,%.4f,%.4f,"
                "%.6e,%.6e,%s\n",
                typeName, M, N, P, BLOCK_DIM * BLOCK_DIM, blocksX, blocksY,
                totalThreads, cElems, t1 - t0, t2 - t1, t3 - t2, maxAbs, maxRel,
                validation.c_str());

    CHECK_CUDA(cudaFree(dA));
    CHECK_CUDA(cudaFree(dB));
    CHECK_CUDA(cudaFree(dC));
    if (!validationPassed) {
        std::fprintf(stderr, "error: validation FAILED (max_abs=%.6e max_rel=%.6e)\n",
                     maxAbs, maxRel);
        return EXIT_FAILURE;
    }
    return EXIT_SUCCESS;
}

int main(int argc, char** argv) {
    if (argc != 1 && argc != 4 && argc != 5) {
        std::fprintf(stderr, "Usage: %s [M N P [double|float]]\n", argv[0]);
        return EXIT_FAILURE;
    }

    size_t M = 128, N = 256, P = 32;
    if (argc > 1 && (!parseDim(argv[1], &M) || !parseDim(argv[2], &N) ||
                     !parseDim(argv[3], &P))) {
        std::fprintf(stderr, "error: dimensions must be positive integers\n");
        return EXIT_FAILURE;
    }
    const std::string type = argc == 5 ? argv[4] : "double";
    if (type == "double") return runCase<double>(M, N, P, type.c_str());
    if (type == "float") return runCase<float>(M, N, P, type.c_str());
    std::fprintf(stderr, "error: invalid type '%s' (use double|float)\n", type.c_str());
    return EXIT_FAILURE;
}


Writing matMul.cu


In [3]:
%%writefile benchmark.py
#!/usr/bin/env python3
"""Benchmark driver for the Q2 naive CUDA matMul.

Runs the compiled matMul executable over an increasing size sweep for both
double and float, stores the raw CSV output per type and renders stacked bar
charts of the H2D / kernel / D2H time breakdown.

Requires: matplotlib (provided by `module load scicomp-python-env` on the
remote cluster). Standard library only otherwise; no Colab dependency.
"""

import argparse
import csv
import subprocess
import sys
from pathlib import Path

HEADER = [
    "type", "M", "N", "P", "threads_per_block", "blocks_x", "blocks_y",
    "total_threads", "active_threads", "H2D_ms", "kernel_ms", "D2H_ms",
    "max_abs_error", "max_rel_error", "validation",
]
TYPES = ("double", "float")
# Increasing sweep; starts at the required 128x256x32 shape and reaches
# 2048^3 cubic work. Deliberately excludes the huge 1024x8191x8197 occupancy
# case, which is run separately (see README).
SWEEP = [
    (128, 256, 32),
    (256, 256, 256),
    (384, 384, 384),
    (512, 512, 512),
    (768, 768, 768),
    (1024, 1024, 1024),
    (1280, 1280, 1280),
    (1536, 1536, 1536),
    (2048, 2048, 2048),
]


def run_case(exe, m, n, p, type_name):
    cmd = [str(exe), str(m), str(n), str(p), type_name]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            f"matMul failed ({' '.join(cmd)}), exit code {proc.returncode}:\n"
            f"{proc.stderr.strip()}"
        )
    lines = [ln for ln in proc.stdout.splitlines() if ln.strip()]
    if len(lines) != 2:
        raise RuntimeError(
            f"unexpected stdout from {' '.join(cmd)}: "
            f"expected 2 lines, got {len(lines)}"
        )
    rows = list(csv.reader(lines))
    if rows[0] != HEADER:
        raise RuntimeError(
            f"CSV header mismatch from {' '.join(cmd)}:\n{rows[0]}"
        )
    if len(rows[1]) != len(HEADER):
        raise RuntimeError(
            f"CSV row has {len(rows[1])} fields, expected {len(HEADER)}"
        )
    val = rows[1][14]
    if val != "full" and not val.startswith("sampled:"):
        raise RuntimeError(f"bad validation field: {val!r}")
    return rows[1]


def write_csv(path, rows):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(rows)


def plot(path, rows, type_name):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    labels = [f"{r[1]}x{r[2]}x{r[3]}" for r in rows]
    h2d = [float(r[9]) for r in rows]
    kernel = [float(r[10]) for r in rows]
    d2h = [float(r[11]) for r in rows]

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.bar(labels, h2d, label="H2D copy (A+B)")
    ax.bar(labels, kernel, bottom=h2d, label="kernel")
    ax.bar(labels, d2h, bottom=[a + b for a, b in zip(h2d, kernel)],
           label="D2H copy (C)")
    ax.set_ylabel("Time (ms)")
    ax.set_xlabel("Matrix shape M x N x P")
    ax.set_title(f"matMul timing breakdown ({type_name}) — 16x16 blocks, "
                 "one thread per C element")
    ax.legend()
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main():
    parser = argparse.ArgumentParser(
        description="Benchmark matMul over an increasing size sweep.")
    parser.add_argument("exe", nargs="?", default="./matMul",
                        help="path to the compiled matMul executable")
    parser.add_argument("--outdir", default="results",
                        help="output directory for CSV/PNG files")
    args = parser.parse_args()

    exe = Path(args.exe).resolve()
    if not exe.exists():
        print(f"error: executable not found: {exe}", file=sys.stderr)
        return 2
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    for type_name in TYPES:
        rows = [run_case(exe, m, n, p, type_name) for m, n, p in SWEEP]
        csv_path = outdir / f"timing_{type_name}.csv"
        png_path = outdir / f"timing_{type_name}.png"
        write_csv(csv_path, rows)
        plot(png_path, rows, type_name)
        print(f"{type_name}: wrote {csv_path} and {png_path}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


Writing benchmark.py


In [4]:
!nvcc -O3 -std=c++17 -arch=sm_75 matMul.cu -o matMul


In [5]:
%%bash
set -eo pipefail
mkdir -p results
./matMul 128 256 32 double | tee results/small_double.csv
./matMul 128 256 32 float | tee results/small_float.csv
./matMul 3 5 7 double
./matMul 1 1 1 float
./matMul 1024 8191 8197 double | tee results/large_double.csv
./matMul 1024 8191 8197 float | tee results/large_float.csv


type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
double,128,256,32,256,2,8,4096,4096,1.0308,1.1848,0.0434,0.000000e+00,0.000000e+00,full
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
float,128,256,32,256,2,8,4096,4096,0.0840,0.2348,0.0293,0.000000e+00,0.000000e+00,full
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
double,3,5,7,256,1,1,256,21,0.0290,0.1188,0.0224,0.000000e+00,0.000000e+00,full
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
float,1,1,1,256,1,1,256,1,0.0280,0.1124,0.0208,0.000000e+00,0.000000e+00,full
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_err

In [6]:
!python3 benchmark.py ./matMul --outdir results


double: wrote results/timing_double.csv and results/timing_double.png
float: wrote results/timing_float.csv and results/timing_float.png


In [7]:
%%bash
set -eo pipefail
ncu --section Occupancy ./matMul 128 256 32 double 2>&1 | tee results/ncu_small_double.log


==PROF== Connected to process 2456 (/content/matMul)
==PROF== Profiling "matMulKernel" - 0: 0%....50%....100% - 1 pass
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
double,128,256,32,256,2,8,4096,4096,0.1740,1198.4824,0.1174,0.000000e+00,0.000000e+00,full
==PROF== Disconnected from process 2456
[2456] matMul@127.0.0.1
  void matMulKernel<double>(const T1 *, const T1 *, T1 *, unsigned long, unsigned long, unsigned long) (2, 8, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           16
    Block Limit Registers                 block            5
    Block Limit Shared Mem                block           16
    Block Limit Warps             

In [8]:
%%bash
set -eo pipefail
ncu --section Occupancy ./matMul 1024 8191 8197 double 2>&1 | tee results/ncu_large_double.log


==PROF== Connected to process 2501 (/content/matMul)
==PROF== Profiling "matMulKernel" - 0: 0%....50%....100% - 1 pass
type,M,N,P,threads_per_block,blocks_x,blocks_y,total_threads,active_threads,H2D_ms,kernel_ms,D2H_ms,max_abs_error,max_rel_error,validation
double,1024,8191,8197,256,513,64,8404992,8393728,135.9443,2414.1758,54.9090,0.000000e+00,0.000000e+00,sampled:81
==PROF== Disconnected from process 2501
[2501] matMul@127.0.0.1
  void matMulKernel<double>(const T1 *, const T1 *, T1 *, unsigned long, unsigned long, unsigned long) (513, 64, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           16
    Block Limit Registers                 block            5
    Block Limit Shared Mem                block           16
    Block

In [9]:
%%bash
set -e
{ nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader
  nvcc --version
  ncu --version
} > results/environment.txt
zip -qr hw1_q2_results.zip results


In [12]:

from google.colab import files
files.download("hw1_q2_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>